# V2 CUDA Test Notebook — S4 shared-memory descriptor tiling

**Người phụ trách:** Khánh — S3 Description và S4 Matching. Notebook và phần kiểm chứng được chuẩn bị với sự hỗ trợ của OpenAI Codex.

V2 kế thừa pipeline V1 đã hoàn thành và đưa vào **một ý tối ưu chính**: tile candidate descriptor trong shared memory ở S4 để nhiều query cùng tái sử dụng. S1–S3, S5–S6, thuật toán, tham số và đường đi dữ liệu giữ như V1. Các stage tiếp tục chạy GPU; không thêm CPU fallback.

Notebook giữ cách trình bày của V1 (setup, build, test, ảnh, bảng và biểu đồ), nhưng tổ chức nội dung theo yêu cầu bản tối ưu: **bottleneck → idea → implementation → result → analysis**. V0 vẫn là reference correctness và speedup; V1 là bản đối chiếu trực tiếp để tách hiệu quả của tiling.

**Cách chạy:** chọn Colab GPU runtime, đưa toàn bộ source V2 mới nhất lên nhánh `v2` hoặc thư mục source trên Colab, rồi chọn **Runtime → Run all**. Lưu notebook kèm toàn bộ output sau khi chạy. Notebook hiện chưa chứa kết quả đo; các phần kết luận bên dưới được tạo từ output thực tế, không điền số liệu giả.

Tài liệu áp dụng: đề Final Project đã cung cấp và `docs/PLAN.md`. Ngưỡng correctness trong notebook là tiêu chí ban đầu của nhóm, cần được chốt bằng V0/V1 và dùng nhất quán; không phải các ngưỡng do đề quy định.

## 1. Setup source, GPU và dataset

Mã nguồn nằm trong `src/` và `tools/v2/` (cùng các test V1 kế thừa trong `tools/`); notebook chỉ điều phối build/run, đọc kết quả và trình bày báo cáo. Không chép implementation C++/CUDA vào cell.

Mặc định lấy source từ nhánh `v2`. Nếu vừa chỉnh source local, phải đưa các thay đổi đó lên GitHub trước khi clone, hoặc upload source vào `REPO_DIR` và bật `USE_EXISTING_SOURCE`. Việc mở riêng notebook không đưa các file `.cu/.cpp` lên Colab.

In [ ]:
from pathlib import Path
import os, sys, shlex, shutil, subprocess, json, uuid, hashlib
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image, Markdown, Code

REPO_URL = 'https://github.com/nganbee/panorama-stitching-project.git'
BRANCH = 'v2'
REPO_DIR = Path('/content/panorama-stitching-project')
USE_EXISTING_SOURCE = False
RUN_INHERITED_TESTS = False

def run(command, check=True, log=None):
    args = shlex.split(command) if isinstance(command, str) else [str(x) for x in command]
    print('$', shlex.join(args), flush=True)
    lines = []
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        status = process.wait()
    if log is not None:
        Path(log).write_text(''.join(lines), encoding='utf-8')
    if check and status != 0:
        raise RuntimeError(f'Command failed with exit code {status}; inspect the output above.')
    return status

if USE_EXISTING_SOURCE:
    assert REPO_DIR.is_dir(), 'Upload source first, then set REPO_DIR.'
elif not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH])
    run(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH])
os.chdir(REPO_DIR)
for directory in ['build', 'out', 'reports']:
    Path(directory).mkdir(exist_ok=True)
required = ['src/stage_implementations/v2_stage4.cu', 'src/cuda/v2_pipeline.cu',
            'tools/v2/verify_pair.cpp', 'tools/v2/benchmark_pair.cpp', 'tools/v2/profile.py']
assert all(Path(p).is_file() for p in required), 'Source is outdated: upload/push the complete updated V2 checkout.'
print('Working directory:', Path.cwd())

In [ ]:
run('nvidia-smi')
run('nvcc --version')
run('g++ --version')
run('git rev-parse HEAD', check=False)
run('git status --short', check=False)
for tool in ['nvcc', 'compute-sanitizer', 'nsys', 'ncu']:
    print(tool, ':', shutil.which(tool) or 'not available')

sites = ['location1', 'location2', 'location3', 'challenge']
sizes = {'small_720p': (1280, 720), 'medium_1080p': (1920, 1080), 'large_4k': (3840, 2160)}
dataset_rows = []
for site in sites:
    for size, expected in sizes.items():
        pair = Path('dataset') / site / size
        for filename in ['1.jpg', '2.jpg']:
            path = pair / filename
            assert path.is_file(), f'Missing input: {path}'
            with PILImage.open(path) as image:
                actual = image.size
            assert actual == expected, (str(path), actual, expected)
        dataset_rows.append({'case': f'{site}_{size}', 'left': str(pair/'1.jpg'),
                             'right': str(pair/'2.jpg'), 'resolution': f'{expected[0]}x{expected[1]}'})
display(pd.DataFrame(dataset_rows))
print('Input pairs to evaluate:', len(dataset_rows))

## 2. Build V0/V1/V2 trên cùng source và cấu hình

`tools/v2/build.sh` biên dịch source hiện tại bằng cùng compiler/flags. Binary production `stitch_v2` chọn version qua CLI; binary đo `benchmark_v2_pair` gọi đúng các hàm pipeline production và lưu từng lần đo. Binary kiểm chứng `verify_v2_pair` chạy riêng, không đưa chi phí kiểm tra vào benchmark.

V2 nằm trong file mới `v2_stage4.cu` và `v2_pipeline.cu`. Các file implementation V1 giữ nguyên. `versions.h` và `report_io.h` chỉ bổ sung đăng ký version. Cờ `PANORAMA_WITH_V2` cho phép build V2 mà không phá lệnh build V1 cũ. Không dùng fast math; giữ `--fmad=false` như các test V1.

Các translation unit `s6_reference.cpp` và `s6_gpu_diagnostics.cu` chỉ phục vụ kiểm chứng: include source S6 gốc để quan sát helper/kernel private. Chúng thay thế object S6 gốc trong **binary verification**, không sửa implementation và không được đưa vào binary benchmark.

In [ ]:
display(Code(Path('tools/v2/build.sh').read_text(encoding='utf-8'), language='bash'))
run('bash tools/v2/build.sh')
run('build/stitch_v2 --list-versions')
params = json.loads(subprocess.check_output(['build/stitch_v2', '--dump-params'], text=True))
display(pd.DataFrame([params]))
assert all(v in params['available_versions'] for v in [0, 1, 2])
print('Build/source registration checks passed. CUDA correctness is checked separately below.')

## 3. Bottleneck — đo V1 và thu bằng chứng profiling

Trước khi đánh giá V2, chạy V0 và V1 trên đủ các cặp/độ phân giải trong cùng GPU session. Dùng kết quả V1 để xác định S4 chiếm bao nhiêu thời gian và đứng ở đâu so với các stage khác. Không suy bottleneck GPU từ timing V0 CPU.

Mỗi case/version có một warm-up và ít nhất năm lần đo. Kết quả chính là **trung bình**, đồng thời lưu raw samples, minimum, maximum, standard deviation và hệ số biến thiên để nhận biết dao động. Stage GPU và copy được đo bằng CUDA events trong pipeline; end-to-end lấy từ đồng hồ host sau đồng bộ, gồm allocation, launch, H2D/D2H và đồng bộ, loại đọc/ghi ảnh. Không cộng kernel/copy vào end-to-end lần nữa.

Mỗi lượt notebook tạo thư mục kết quả riêng, lưu GPU UUID, boot/session, compiler, commit, hash source/binary và dataset. Giai đoạn V2 kiểm tra lại metadata để ngăn trộn số đo khác session/source. V0/V1 được đo lại trong phiên này dù V1 đã nghiệm thu trước đó.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = Path('reports/v2_benchmark') / RUN_ID
print('Result directory:', RUN_DIR)
run([sys.executable, 'tools/v2/benchmark.py', '--phase', 'baseline', '--run-dir', RUN_DIR])
baseline = pd.read_csv(RUN_DIR / 'summary.csv')
display(baseline.reindex(columns=['case', 'status', 'v0_match_ms', 'v1_match_ms',
                                  'v1_kernel_total_ms', 'v1_h2d_ms', 'v1_d2h_ms',
                                  'v1_end_to_end_ms', 'v1_S4_kernel_share']))

In [ ]:
stage_columns = [f'v1_S{i}_ms' for i in range(1, 7)]
stage_table = baseline.set_index('case').reindex(columns=stage_columns).dropna(how='all')
if not stage_table.empty:
    stage_table.columns = [f'S{i}' for i in range(1, 7)]
    display(stage_table)
    ranked = stage_table.rank(axis=1, ascending=False, method='min')
    display(pd.DataFrame({'largest_stage': stage_table.idxmax(axis=1), 'S4_rank': ranked['S4']}))
    ax = stage_table.plot.bar(stacked=True, figsize=(14, 5))
    ax.set_ylabel('V1 GPU kernel time (ms)')
    ax.set_title('V1 per-stage kernel breakdown — evidence before evaluating V2')
    plt.tight_layout(); plt.show()
else:
    print('No successful V1 baseline case. Inspect raw logs before selecting an optimization.')

### Bằng chứng profiler

Theo yêu cầu, bảng timing chưa thay thế được output profiler. Cell dưới ưu tiên Nsight Systems để thu timeline/tổng hợp kernel và copy. Nếu không chạy được, script thử Nsight Compute trên các kernel nearest-neighbour của S4. Chọn case có S4 V1 lâu nhất trong baseline để chẩn đoán, lưu case, GPU, câu lệnh, raw report và bản tóm tắt text/CSV.

**Đọc output rồi mới kết luận:** với Systems, đối chiếu `nearest_kernel` với tổng kernel/copy; với Compute, dùng các counter được thu để giải thích kernel S4, kết hợp bảng thời gian toàn pipeline. Report được thu thành công chưa tự chứng minh S4 là bottleneck lớn nhất.

Profiler có thể không cài hoặc bị chặn counter/CUPTI trên Colab. Khi đó notebook ghi **NOT VERIFIED**, giữ log và vẫn cho chạy kiểm chứng/benchmark; phần báo cáo profiling còn thiếu cho đến khi có bằng chứng thực tế từ môi trường hỗ trợ. Không tạo screenshot/text giả. Timing của lần chạy dưới profiler nằm riêng trong `profiling/` và không dùng tính speedup.

In [ ]:
run([sys.executable, 'tools/v2/profile.py', '--run-dir', RUN_DIR, '--version', '1'])
profile_v1 = json.loads((RUN_DIR/'profiling/profile_v1.json').read_text(encoding='utf-8'))
print('V1 profiling status:', profile_v1['status'])
if profile_v1['status'] == 'collected':
    print('Case:', profile_v1['case'], '| tool:', profile_v1['tool'])
    print('Raw report:', profile_v1['report'])
    print(Path(profile_v1['summary']).read_text(encoding='utf-8'))
else:
    display(Markdown('**Thiếu bằng chứng profiling.** Không xem bước này là PASS; xem các log trong `profiling/`.'))

## 4. Idea và implementation — chỉ thay cách đọc candidate ở S4

### Pipeline kế thừa

```text
RGB A/B → S1 gray/blur [V1] → S2 Harris/NMS/top-K [V1]
        → S3 BRIEF [V1] → S4 shared-memory descriptor tiling [V2]
        → S5 RANSAC/refinement [V1] → S6 warp/blend/crop [V1] → Panorama
```

### V1 và V2 khác nhau ở đâu?

| Thành phần | V1 | V2 |
|---|---|---|
| Phân công query | Một thread/query | Giữ nguyên |
| Candidate descriptor | Đọc global memory trong vòng quét | Block cùng nạp từng tile rồi đọc shared memory |
| Block/tile | Block baseline | 128 thread, tile 128 descriptor, mỗi descriptor 256 bit; 4 KB shared memory |
| Hamming | XOR + popcount trên bốn word 64 bit | Giữ nguyên |
| Top-2/tie-break | Duyệt index tăng dần | Giữ thứ tự tăng dần xuyên qua các tile |
| Ratio/cross-check/compaction | Quy tắc V1 | Giữ nguyên, gồm thứ tự output |
| Các stage và H2D/D2H khác | V1 | Giữ nguyên |

Mỗi thread giữ query của mình, còn các thread trong block cùng tải candidate vào shared memory. Sau barrier đầu, các query hợp lệ quét tile để cập nhật best/second-best. Barrier thứ hai bảo đảm đọc xong trước khi tile kế tiếp ghi đè. Thread vượt số query ở block cuối **vẫn đi qua cả hai barrier**, chỉ bỏ phần tính/output. Tile cuối chỉ đọc các phần tử đã nạp.

Tiling tạo khả năng tái sử dụng candidate giữa các query. Đây không phải cam kết giảm đúng một hệ số cố định số byte DRAM: V1 đã được hỗ trợ bởi cache GPU. Shared-memory load và đồng bộ có chi phí; với ít descriptor hoặc cache tốt, V2 có thể chậm hơn. V2 không gộp SoA, warp/query, đổi precision hoặc giữ dữ liệu liên tục trên device vào cùng thí nghiệm.

In [ ]:
v2_source = Path('src/stage_implementations/v2_stage4.cu').read_text(encoding='utf-8')
begin = v2_source.index('// All 128 threads')
end = v2_source.index('__global__ void filter_kernel', begin)
display(Code(v2_source[begin:end], language='cuda'))
print('Implementation file:', 'src/stage_implementations/v2_stage4.cu')
print('Source SHA256:', hashlib.sha256(v2_source.encode()).hexdigest())

## 5. Kiểm chứng riêng S4 và kiểm tra CUDA

### Parity trên cùng descriptor đầu vào

V0 là oracle tuần tự. Test gọi cả `v1_match` và `v2_match` trên cùng hai mảng descriptor, so **số lượng, thứ tự và từng trường `idxA`, `idxB`, `dist`**, không chỉ đếm matches. Bao gồm input rỗng, một candidate không có second neighbour, descriptor trùng/tie, kích thước hai phía khác nhau, tile đầy/thiếu và biên tile. Descriptor BRIEF V1 cũng được đối chiếu với V0 trên cùng image/keypoints.

Các test này kiểm tra semantics độc lập với sai số của S1–S2. Ở bước ảnh thật phía sau, verifier tiếp tục lấy descriptor đã trích xuất để chạy cả ba matcher trên cùng input.

In [ ]:
run('build/stage34_v2_selftest')

### Memory check và synchronization check

Memory check kiểm tra truy cập bộ nhớ device; synccheck tập trung vào lỗi đồng bộ barrier của tile. Đây là kiểm tra tính hợp lệ khi thực thi, không thay thế parity và không dùng đo hiệu năng. `--error-exitcode 1` giúp lỗi được phản ánh thành lệnh thất bại. Nếu công cụ thiếu, trạng thái là chưa kiểm chứng, không phải PASS.

In [ ]:
sanitizer_results = {}
if shutil.which('compute-sanitizer'):
    for tool in ['memcheck', 'synccheck']:
        status = run(['compute-sanitizer', '--tool', tool, '--error-exitcode', '1',
                      'build/stage34_v2_selftest'], check=False,
                     log=RUN_DIR/f'S4_{tool}.log')
        sanitizer_results[tool] = status == 0
else:
    sanitizer_results = {'memcheck': False, 'synccheck': False}
    print('NOT VERIFIED: compute-sanitizer is not available.')
print('S4 sanitizer results:', sanitizer_results)

### Kiểm tra kế thừa từ V1 — tuỳ chọn

V1 đã được nghiệm thu; không cần biến notebook V2 thành một lần phát triển lại S1–S6. Có thể bật `RUN_INHERITED_TESTS` khi đổi toolchain hoặc cần điều tra lỗi. Các test stage S1–S2, S5 và S6 dùng nguyên file V1. Dù tắt mục này, kiểm chứng toàn pipeline trên ảnh thật ở dưới vẫn chạy. Timing harness S1–S2 V0 riêng đã bỏ khỏi notebook vì không đánh giá tối ưu S4.

In [ ]:
if RUN_INHERITED_TESTS:
    inherited = [
        ('stage12_cuda_selftest', ['tools/run_stage12_cuda_selftest.cpp',
            'src/stage_implementations/v0_stage12.cpp', 'src/stage_implementations/v1_stage12.cu']),
        ('stage5_cuda_selftest', ['tools/test_v1_stage5.cu', 'src/stage_implementations/v0_stage5.cpp']),
        ('test_v1_stage6', ['tools/test_v1_stage6.cpp', 'src/stage_implementations/v0_stage6.cpp',
            'src/stage_implementations/v1_stage6.cu']),
    ]
    for executable, sources in inherited:
        run(['nvcc', '-O2', '-std=c++17', '--fmad=false', '-Isrc/core_headers',
             *sources, '-o', f'build/{executable}'])
        run([f'build/{executable}'])
else:
    print('Inherited standalone V1 tests skipped by configuration; full V2 verification remains required.')

## 6. Kiểm chứng tích hợp và panorama trên ảnh thật

Test tổng hợp kiểm tra identity/dịch chuyển, input hỏng và ảnh không có feature. Sau đó verifier chạy một cặp ảnh thật để kiểm tra đường đi đầy đủ trước khi đánh giá toàn dataset. Chi phí verification và copy mask chỉ nằm ở lượt kiểm tra riêng, không nằm trong timing benchmark.

### Tiêu chí theo PLAN của nhóm

| Đầu ra | Điều kiện |
|---|---|
| Gray/blur/Harris | Cùng input, `abs_error ≤ atol + rtol × abs(V0)`; lần lượt `(1e-4,1e-6)`, `(2e-4,2e-6)`, `(5e-2,2e-5)` |
| Keypoints | Cùng số lượng, tọa độ, thứ tự; score trong dung sai Harris |
| Descriptors | Cùng input phải trùng toàn bộ bit; kiểm tra cả chuỗi trích xuất thực tế |
| Matches | Cùng số lượng, thứ tự, `idxA`, `idxB`, `dist`; so cả same-input và end-to-end |
| S5 | Cùng inlier sau refinement; so H bằng sai số chiếu B→A trên keypoints và lưới phủ ảnh B, tối đa `1e-3 px` |
| S6 | Canvas, origin, valid mask, vị trí/kích thước crop trùng; panorama cùng shape, sai số tối đa 1 mức/kênh; báo cáo MAE |
| V1/V2 trực tiếp | Matches, phép chiếu H và panorama phải khớp chính xác vì chỉ thay cách đọc descriptor S4 |
| Ca lỗi | Ghi đúng status; không CPU fallback; hai bản cùng thất bại không được ghi panorama correctness PASS |

Các ngưỡng là hợp đồng nghiệm thu ban đầu của nhóm. Nếu cần sửa, phải có dữ liệu V0/V1 giải thích và kiểm tra lại các version, không nới riêng V2. Không chỉ so hệ số H hoặc nhìn ảnh gần giống. Verifier replay kernel S6 gốc để lấy valid mask và crop, so với reference CPU; không suy valid mask từ RGB đen.

In [ ]:
run('build/test_v2_pipeline')
LEFT = 'dataset/location1/small_720p/1.jpg'
RIGHT = 'dataset/location1/small_720p/2.jpg'
SMOKE_REPORT = RUN_DIR/'smoke_verification.json'
smoke_status = run(['build/verify_v2_pair', LEFT, RIGHT, SMOKE_REPORT], check=False,
                   log=RUN_DIR/'smoke_verification.log')
smoke = json.loads(SMOKE_REPORT.read_text(encoding='utf-8'))
print('Smoke verification:', smoke['status'])
display(pd.DataFrame([{'check': key, 'passed': value} for key,value in smoke['checks'].items()]))
display(pd.DataFrame([smoke.get('metrics', {})]).T.rename(columns={0:'value'}))
if smoke['status'] != 'pass':
    print('NOT PASS: inspect checks/log; later cases are still recorded for failure analysis.')

## 7. Benchmark V2 và kiểm chứng đủ dataset

Chạy V2 trong chính session/source/binary/input của baseline. Mỗi case được kiểm chứng riêng bằng `verify_v2_pair`, sau đó đo pipeline production với cùng warm-up/số lần như V0/V1. Mọi case đều được lưu report và log, kể cả khi cặp khó thất bại.

`consistent_pipeline_failure` nghĩa là các bản đều chưa tạo được panorama; phải phân tích nguyên nhân, không phải correctness PASS. `pipeline_status_mismatch`, `correctness_failed`, `process_error` và `error` cần điều tra. Không tính speedup hợp lệ cho case chưa có output được kiểm chứng. Một case lỗi không được làm mất kết quả của các case còn lại.

In [ ]:
benchmark_status = run([sys.executable, 'tools/v2/benchmark.py', '--phase', 'optimized',
                        '--run-dir', RUN_DIR], check=False)
summary = pd.read_csv(RUN_DIR/'summary.csv')
display(summary.reindex(columns=['case', 'status', 'verification_status', 'failed_checks']))
print('Benchmark process exit status:', benchmark_status)
print('All raw reports/logs:', RUN_DIR)

## 8. Result — correctness, timing và speedup

### Correctness theo từng case

Bảng dưới lấy trực tiếp từ verification JSON: sai số chiếu H, sai số panorama, MAE, số pixel mask khác nhau và các check thất bại. Các case thiếu kết quả vẫn giữ trong bảng trạng thái. Đây là bằng chứng số, bổ sung cho ảnh minh hoạ.

In [ ]:
verification_columns = ['case', 'status', 'failed_checks', 'V0_V2_projection_error_px',
    'V0_V2_pixel_max_error', 'V0_V2_pixel_mae', 'V0_V2_mask_mismatch_pixels',
    'V1_V2_projection_error_px', 'V1_V2_pixel_max_error', 'V1_V2_pixel_mae',
    'V1_V2_mask_mismatch_pixels']
display(summary.reindex(columns=verification_columns))
feature_columns = ['case', 'status'] + [f'v{v}_{key}' for v in [0,1,2] for key in ['keypoints_A','keypoints_B','matches','inliers','panorama_width','panorama_height']]
display(summary.reindex(columns=feature_columns))


### Bảng so sánh như V1, mở rộng thêm V2

Đơn vị thời gian là ms. S1–S3 cộng thời gian xử lý hai ảnh; S4–S6 xử lý quan hệ giữa hai ảnh. Stage GPU là kernel-only, còn end-to-end bao gồm overhead. Vì vậy, speedup S4 thể hiện hiệu quả phần tính toán matching; speedup end-to-end mới thể hiện lợi ích toàn pipeline.

`Speedup V0/V2 = T_V0 / T_V2`; `Speedup V1/V2 = T_V1 / T_V2`. Giá trị lớn hơn 1 là nhanh hơn; bản chậm hơn vẫn được báo cáo và phân tích. Không trộn số đo dưới profiler/sanitizer vào các bảng này.

In [ ]:
valid = summary[summary.status == 'pass'].copy()
speedup_columns = ['case', 'v0_match_ms', 'v1_match_ms', 'v2_match_ms',
    'v0_over_v2_match_speedup', 'v1_over_v2_match_speedup',
    'v0_end_to_end_ms', 'v1_end_to_end_ms', 'v2_end_to_end_ms',
    'v0_over_v2_end_to_end_speedup', 'v1_over_v2_end_to_end_speedup']
display(valid.reindex(columns=speedup_columns))
stage_rows = []
for _, row in summary.iterrows():
    for stage in ['S1','S2','S3','S4','S5','S6','kernel_total','h2d','d2h','end_to_end']:
        entry = {'case':row['case'], 'status':row['status'], 'stage':stage}
        for version in [0,1,2]:
            entry[f'V{version} mean (ms)'] = row.get(f'v{version}_{stage}_ms', np.nan)
        if row['status'] == 'pass' and stage in ['S1','S2','S3','S4','S5','S6','end_to_end']:
            denominator = entry['V2 mean (ms)']
            if denominator > 0:
                entry['V0/V2'] = entry['V0 mean (ms)']/denominator
                entry['V1/V2'] = entry['V1 mean (ms)']/denominator
        stage_rows.append(entry)
stage_comparison = pd.DataFrame(stage_rows)
display(stage_comparison)
stage_comparison.to_csv(RUN_DIR/'stage_comparison.csv', index=False)

In [ ]:
dispersion_rows = []
for _, row in summary.iterrows():
    for version in [0,1,2]:
        for metric in ['S1','S2','S3','S4','S5','S6','kernel_total','h2d','d2h','end_to_end']:
            dispersion_rows.append({'case':row['case'], 'version':version, 'metric':metric,
                'mean_ms':row.get(f'v{version}_{metric}_ms'),
                'minimum_ms':row.get(f'v{version}_{metric}_min_ms'),
                'maximum_ms':row.get(f'v{version}_{metric}_max_ms'),
                'std_ms':row.get(f'v{version}_{metric}_std_ms'),
                'cv':row.get(f'v{version}_{metric}_cv')})
dispersion = pd.DataFrame(dispersion_rows)
display(dispersion[dispersion.metric.isin(['S4','end_to_end'])])
dispersion.to_csv(RUN_DIR/'dispersion.csv', index=False)
print('Full dispersion table:', RUN_DIR/'dispersion.csv')
print('CV is standard deviation / mean; inspect raw samples before interpreting small differences.')

### Biểu đồ theo case và độ phân giải

Biểu đồ giữ các case hợp lệ, không che bản chậm hơn. Bảng trạng thái ở trên vẫn liệt kê toàn bộ dataset. Breakdown và speedup được vẽ riêng để tránh nhầm kernel-only với end-to-end; chỉ cộng kernel/H2D/D2H trong biểu đồ thành phần, không cộng thêm end-to-end.

In [ ]:
if not valid.empty:
    indexed = valid.set_index('case')
    for metric in ['match','end_to_end']:
        columns = [f'v{v}_{metric}_ms' for v in [0,1,2]]
        ax = indexed[columns].plot.bar(figsize=(14,5), logy=True)
        ax.set_ylabel('Mean time (ms, logarithmic scale)')
        ax.set_title(f'V0 / V1 / V2 — {metric}')
        plt.tight_layout(); plt.show()
    ax = indexed[['v1_over_v2_match_speedup','v1_over_v2_end_to_end_speedup']].plot.bar(figsize=(14,5))
    ax.axhline(1,color='black',linestyle='--')
    ax.set_ylabel('V1 time / V2 time'); ax.set_title('Effect of S4 tiling by case and resolution')
    plt.tight_layout(); plt.show()
    parts = indexed[[f'v2_S{i}_ms' for i in range(1,7)]+['v2_h2d_ms','v2_d2h_ms']]
    ax = parts.plot.bar(stacked=True,figsize=(14,5))
    ax.set_ylabel('Measured component time (ms)')
    ax.set_title('V2 kernel/copy breakdown; allocation/host gaps are not shown')
    plt.tight_layout(); plt.show()
    resolution_rows = []
    for size in sizes:
        subset = valid[valid.case.str.endswith(size)]
        resolution_rows.append({'resolution':size, 'valid_cases':len(subset),
            'mean_case_speedup_S4_V1_V2':subset['v1_over_v2_match_speedup'].mean(),
            'mean_case_speedup_E2E_V1_V2':subset['v1_over_v2_end_to_end_speedup'].mean()})
    display(pd.DataFrame(resolution_rows))
    print('Resolution averages are averages of per-case speedup ratios; individual cases remain above.')
else:
    print('No verified case: speedup plots are not available. Inspect correctness and failure logs.')

### Panorama gallery và failure analysis

Hiển thị kết quả V2 kèm panorama V0/V1 trên cùng case để kiểm tra trực quan. Ảnh chỉ minh hoạ, không thay các phép so bit/match/geometry/pixel. Các cặp thất bại phải giữ log để giải thích thiếu texture, pattern lặp, hình học hoặc lỗi thực thi; không tự gán nguyên nhân chỉ từ status.

In [ ]:
for _, row in summary.iterrows():
    display(Markdown(f"**{row['case']} — {row['status']}**"))
    for version in [0,1,2]:
        report_path = RUN_DIR/f"{row['case']}_v{version}.json"
        report = json.loads(report_path.read_text(encoding='utf-8')) if report_path.is_file() else {}
        path = Path(report.get('panorama',{}).get('path',''))
        if report.get('status') == 'ok' and path.is_file():
            print(f'V{version}:', path)
            with PILImage.open(path) as original:
                preview = original.copy()
            preview.thumbnail((800, 450))
            display(preview)
        else:
            print(f'V{version} unavailable:', report.get('error','missing report'))
    if row['status'] != 'pass':
        log = RUN_DIR/f"{row['case']}_verification.log"
        if log.is_file():
            print(log.read_text(encoding='utf-8')[-6000:])

### Profiling V2 để giải thích kết quả — lượt chạy riêng

Lặp lại chẩn đoán trên cùng case mà script đã chọn từ V1. So các kernel nearest V1/V2, resource/counter có sẵn và chi phí copy. Sự khác biệt của profiling hỗ trợ giải thích speedup, nhưng số thời gian chính vẫn lấy từ các lượt benchmark không có profiler.

In [ ]:
run([sys.executable, 'tools/v2/profile.py', '--run-dir', RUN_DIR, '--version', '2'])
profile_v2 = json.loads((RUN_DIR/'profiling/profile_v2.json').read_text(encoding='utf-8'))
print('V2 profiling status:', profile_v2['status'])
if profile_v2['status'] == 'collected':
    print('Report:', profile_v2['report'])
    print(Path(profile_v2['summary']).read_text(encoding='utf-8'))

## 9. Analysis — nhận xét từ kết quả đã chạy

Phân tích tách hai câu hỏi: tile có giảm thời gian S4 không, và việc đó có giúp toàn pipeline không? Stage khác giữ code V1 nên thay đổi thời gian của chúng là dao động hoặc ảnh hưởng thực thi, không được nhận là tối ưu mới của V2.

Nếu S4 nhanh hơn nhưng end-to-end ít cải thiện, đối chiếu tỷ trọng S4, H2D/D2H và host overhead. Nếu S4 chậm hơn, dùng profiler để kiểm tra cache, barrier, resource usage và quy mô descriptor; không khẳng định nguyên nhân từ thời gian một mình. Dữ liệu nhỏ có thể không bù được overhead. Bản chậm hơn vẫn là kết quả hợp lệ nếu correctness đạt và có phân tích trung thực.

Cell dưới tạo các nhận xét định lượng từ output; chúng không thay phần giải thích bằng bằng chứng profiler. Mỗi nhận xét trong slide/báo cáo phải truy được về bảng, log hoặc report đã lưu.

In [ ]:
if not valid.empty:
    faster = valid[valid.v1_over_v2_match_speedup > 1]
    slower = valid[valid.v1_over_v2_match_speedup < 1]
    e2e_faster = valid[valid.v1_over_v2_end_to_end_speedup > 1]
    display(Markdown(
        f"Có **{len(valid)} / {len(summary)}** case có panorama được kiểm chứng. "
        f"S4 V2 nhanh hơn V1 ở **{len(faster)}** case, chậm hơn ở **{len(slower)}** case; "
        f"end-to-end nhanh hơn ở **{len(e2e_faster)}** case. "
        f"Speedup S4 V1/V2 nằm trong **{valid.v1_over_v2_match_speedup.min():.3f}–"
        f"{valid.v1_over_v2_match_speedup.max():.3f}×**; end-to-end trong "
        f"**{valid.v1_over_v2_end_to_end_speedup.min():.3f}–"
        f"{valid.v1_over_v2_end_to_end_speedup.max():.3f}×**."))
    display(slower.reindex(columns=speedup_columns))
    display(Markdown('Các nguyên nhân về cache/barrier cần đối chiếu report profiler trước khi viết kết luận.'))
failed = summary[summary.status != 'pass']
display(failed.reindex(columns=['case','status','failed_checks','v0_error','v1_error','v2_error']))
if not failed.empty:
    print('Investigate each failure; equal pipeline failure is not panorama correctness PASS.')

### Bố cục đưa vào báo cáo và slides

| Phần theo đề | Nội dung có thể trích từ notebook |
|---|---|
| a. Bottleneck | Mục 3: bảng V1, tỷ trọng/rank S4, output profiler kèm case/GPU |
| b. Idea | Mục 4: candidate reuse trong shared memory và chi phí đồng bộ |
| c. Implementation | Mục 4–5: kernel thật, mapping, tile cuối, matching invariants |
| d. Result | Mục 6–8: parity, mask/crop, sai số chiếu, bảng/biểu đồ speedup, gallery |
| e. Analysis | Mục 9: nhanh/chậm theo case, khác biệt S4/end-to-end, giới hạn và nguyên nhân có bằng chứng |

Không viết kết luận rằng V2 chắc chắn nhanh hơn. Khi profiler hoặc correctness chưa đạt, ghi rõ phần thiếu thay vì kết luận hoàn tất. Với failure case, bổ sung giải thích dựa trên ảnh/matches/homography và log cụ thể. Phần đóng góp ghi Khánh phụ trách S4 V2 và kiểm chứng/benchmark; các stage kế thừa ghi đúng tác giả nhóm.

## 10. Trạng thái nghiệm thu, file bàn giao và nguồn tham khảo

Notebook lưu cả các trạng thái thiếu để người đọc biết điều gì đã thực sự được kiểm chứng. Với Colab không hỗ trợ profiler/sanitizer, các bước đó chưa được xác nhận dù benchmark có thể hoàn thành. Không sửa output bằng tay để biến lỗi thành PASS.

File bàn giao gồm notebook đã lưu output, source V2 và test, thư mục kết quả của `RUN_ID`, raw profiling report, JSON/CSV/log cùng panorama. `summary.csv`, `stage_comparison.csv` và `dispersion.csv` phục vụ tổng hợp; chúng không thay raw samples và log. Lưu kết quả ra nơi bền vững trước khi Colab mất session.

In [ ]:
checklist = pd.DataFrame([
    {'item':'S4 memcheck', 'complete':sanitizer_results.get('memcheck',False)},
    {'item':'S4 synccheck', 'complete':sanitizer_results.get('synccheck',False)},
    {'item':'V1 profiling collected', 'complete':profile_v1['status']=='collected'},
    {'item':'V2 profiling collected', 'complete':profile_v2['status']=='collected'},
    {'item':'All dataset panoramas verified', 'complete':len(valid)==len(dataset_rows)},
    {'item':'Raw samples and comparison tables saved',
     'complete':all((RUN_DIR/p).is_file() for p in ['summary.csv','stage_comparison.csv','dispersion.csv'])},
])
display(checklist)
print('Result directory to preserve:', RUN_DIR)
print('Pending:', checklist.loc[~checklist.complete, 'item'].tolist())
print('Also required: interpret actual profiler output, explain failure/slower cases, and save this notebook with outputs.')

### Nguồn và khai báo AI

- Đề `CSC14120_2026_Final Project.pdf` do giảng viên cung cấp: version kế thừa, đo công bằng, correctness, profiling và cấu trúc báo cáo.
- `docs/PLAN.md`: phân công Khánh, scope S4 V2 và hợp đồng correctness của nhóm.
- Source V0/V1 của nhóm: thuật toán, tham số, BRIEF pattern, matching và pipeline được kế thừa. Header các file mới ghi tác giả và nguồn.
- [NVIDIA Nsight Systems — User Guide](https://docs.nvidia.com/nsight-systems/UserGuide/index.html): lệnh capture profile.
- [NVIDIA Nsight Systems — Analysis Guide](https://docs.nvidia.com/nsight-systems/AnalysisGuide/): báo cáo tổng hợp kernel/copy.
- [NVIDIA Nsight Compute — CLI](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html): thu và xuất counter cho kernel S4.

**Khai báo AI:** OpenAI Codex được dùng để hỗ trợ tổ chức notebook, cài đặt descriptor tiling, xây dựng kiểm chứng và script tổng hợp. Thành viên phải đọc, giải thích, sửa được code và tự chạy xác nhận các output khi bảo vệ. Thời gian, ảnh và profiler evidence phải sinh từ source nộp; không tạo số liệu bằng AI.